# Notebook 03 – Network Building
## Echo Chambers in Climate Change Discussions on Reddit

Builds NetworkX graphs from the cleaned edge DataFrame.

### What this notebook produces:
- **Monthly graphs** – one per calendar month (for temporal tracking)
- **Quarterly graphs** – for smoother trends
- **Event graphs** – before/after windows around major climate events
- **Full graph** – entire dataset merged (for EDA / overview)

All graphs are saved as `GraphML` files in `data/processed/graphs/`.

In [ ]:
import sys, pickle
sys.path.insert(0, '..')

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from src.network_builder import NetworkBuilder, CLIMATE_EVENTS
from src.metrics import GraphMetrics

GRAPH_DIR = Path('../data/processed/graphs')
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
print("✅ Imports successful")

## Step 1: Load Cleaned Data

In [ ]:
df = pd.read_parquet('../data/processed/comments_clean.parquet')
print(f"Loaded: {len(df):,} edges")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['datetime'].min()} → {df['datetime'].max()}")
df.head(3)

## Step 2: Build the Full Graph (Overview)

In [ ]:
builder = NetworkBuilder(df, directed=False)  # Undirected for Louvain

G_full = builder.build_full_graph()
print(f"\nFull graph stats:")
print(f"  Nodes    : {G_full.number_of_nodes():,}")
print(f"  Edges    : {G_full.number_of_edges():,}")
print(f"  Density  : {nx.density(G_full):.6f}")

# Largest connected component
G_lcc = NetworkBuilder.largest_component(G_full)
print(f"\nLargest Connected Component:")
print(f"  Nodes    : {G_lcc.number_of_nodes():,}")
print(f"  Edges    : {G_lcc.number_of_edges():,}")
print(f"  % of full: {100*G_lcc.number_of_nodes()/G_full.number_of_nodes():.1f}%")

In [ ]:
# Degree distribution of full graph
degrees = [d for _, d in G_full.degree()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0f1117')
for ax in [ax1, ax2]:
    ax.set_facecolor('#1a1d27')
    for spine in ax.spines.values(): spine.set_color('#444')
    ax.tick_params(colors='#aaa')

# Linear scale
ax1.hist(degrees, bins=50, color='#00e5ff', alpha=0.8, edgecolor='none')
ax1.set_title('Degree Distribution', color='white')
ax1.set_xlabel('Degree', color='#aaa'); ax1.set_ylabel('Count', color='#aaa')

# Log-log (power-law check)
deg_counts = pd.Series(degrees).value_counts().sort_index()
ax2.loglog(deg_counts.index, deg_counts.values, 'o', color='#ff6b6b', markersize=3, alpha=0.7)
ax2.set_title('Degree Distribution (log-log)', color='white')
ax2.set_xlabel('Degree', color='#aaa'); ax2.set_ylabel('Count', color='#aaa')
ax2.grid(color='#2d2d3a', alpha=0.5)

plt.suptitle('Full Network Degree Distribution', color='white', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/degree_distribution.png', dpi=120, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## Step 3: Build Monthly Graphs

In [ ]:
monthly_graphs = builder.build_monthly_graphs()

print(f"\nBuilt {len(monthly_graphs)} monthly graphs")
print("\nSample stats:")
print(f"{'Period':<12} {'Nodes':>8} {'Edges':>8} {'Density':>10}")
print("-" * 42)
for label, G in sorted(monthly_graphs.items()):
    print(f"{label:<12} {G.number_of_nodes():>8,} {G.number_of_edges():>8,} {nx.density(G):>10.6f}")

In [ ]:
# Save monthly graphs as pickle (preserves all attributes)
with open(GRAPH_DIR / 'monthly_graphs.pkl', 'wb') as f:
    pickle.dump(monthly_graphs, f)
print(f"✅ Saved monthly graphs → {GRAPH_DIR / 'monthly_graphs.pkl'}")

## Step 4: Build Event-Based Before/After Graphs

In [ ]:
print("Climate events defined:")
for name, date in CLIMATE_EVENTS.items():
    print(f"  {name:<35s} → {date}")

In [ ]:
event_graphs = builder.build_event_graphs(window_days=60)

print(f"\nEvent graphs built: {len(event_graphs)}")
for event, windows in event_graphs.items():
    b = windows['before']
    a = windows['after']
    print(f"\n  {event}")
    print(f"    Before: {b.number_of_nodes():,} nodes, {b.number_of_edges():,} edges")
    print(f"    After : {a.number_of_nodes():,} nodes, {a.number_of_edges():,} edges")

In [ ]:
with open(GRAPH_DIR / 'event_graphs.pkl', 'wb') as f:
    pickle.dump(event_graphs, f)
print(f"✅ Saved event graphs → {GRAPH_DIR / 'event_graphs.pkl'}")

## Step 5: Quick Community Detection Test (on 1 month)

In [ ]:
# Pick a month with decent data
test_label = sorted(monthly_graphs.keys())[len(monthly_graphs)//2]  # middle month
G_test = monthly_graphs[test_label]

gm = GraphMetrics(betweenness_k=50)
stats = gm.compute_all(G_test, label=test_label)

print(f"\n{'─'*45}")
print(f"  Test period       : {test_label}")
print(f"  Nodes             : {stats['num_nodes']:,}")
print(f"  Edges             : {stats['num_edges']:,}")
print(f"  Modularity        : {stats['modularity']:.4f}")
print(f"  Communities       : {stats['num_communities']}")
print(f"  Assortativity     : {stats['degree_assortativity']:.4f}")
print(f"  Avg Clustering    : {stats['avg_clustering']:.4f}")
print(f"{'─'*45}")

## ✅ Done!
All graphs built and saved. Proceed to **`04_temporal_analysis.ipynb`**.